# Notebook for using the tool with template data

## Import libraries and modules

In [ ]:
import spectfbcalc_lib as sfc
from climtools import climtools_lib as ctl 
import output_lib as out

In [ ]:
from importlib import reload
reload(sfc)

In [ ]:
# Test libraries import

sfc.mytestfunction()

In [ ]:
ctl.datestamp()

In [ ]:
import sys
import os
import glob

import numpy as np
import xarray as xr

from matplotlib import pyplot as plt
import matplotlib.cbook as cbook

### If spectral kernels are used

In [ ]:
from dask_jobqueue import SLURMCluster
from dask.distributed import Client

# Dask will automatically submit SLURM jobs for you
cluster = SLURMCluster(
    cores=4,
    memory="64GB",
    processes=4,
    walltime="01:00:00",
    #qos="np",
    #account='spitfabi',
    #interface='ib0'  # or 'eth0', depends on your HPC
    job_extra_directives=[
        "--account=name-of-your-account",
        "--qos=np"
        # "--constraint=haswell",
        # "--exclusive",
        # "--mail-type=END,FAIL",
        # "--mail-user=your.email@domain.com"
    ]
)

# Scale to desired number of workers
cluster.scale(jobs=4)  # This submits 4 SLURM jobs

# Connect client
client = Client(cluster)

In [ ]:
print(client.dashboard_link)

In [ ]:
print(client)

In [ ]:
import dask.array as da
x = da.random.random((20000, 20000), chunks=(1000, 1000))
result = (x + x.T).mean().compute()
print(result)

# To check the status of the workers and the number of tasks executed, you can use the following code:
info = client.scheduler_info()['workers']
for addr, w in info.items():
    print(addr, "- tasks:", w.get('metrics', {}).get('task_counts', 'n/a'))

## Experiment setup

In [ ]:
# Load the configuration file and preprocess the data
config='config_template.yaml'
ker='HUANG' # or 'ERA5' or 'SPECTRAL'
control, experiment, kernel = sfc.preprocess_data(config, ker)

In [ ]:
# OR: Load the control experiment object directly, without preprocessing the data
import yaml
with open('config_template.yaml', 'r') as f:
    config_dict = yaml.safe_load(f)

ker = 'HUANG'  # or 'ERA5' or 'SPECTRAL'
raw_variables = {"hus", "rlut", "rsdt", "rlutcs", "alb", "rsut", "rsutcs", "ta", "tas", "ts"}
control = sfc.Experiment('PI', config_dict['file_paths']['reference_dataset'], remap_dir = config_dict['file_paths']['output'] + f"remapped_{ker}/", raw_variables = raw_variables, variable_mapping = config_dict['variable_mapping'])

## Compute anomalies

In [ ]:
# Compute anomalies one by one and save them in the specified folder
cart_out='./output/'
sfc.Rad_anomaly_planck_atm_lr(experiment,  kernel, cart_out)
sfc.Rad_anomaly_wv(experiment, control, kernel, cart_out)
sfc.Rad_anomaly_albedo(experiment, kernel, cart_out)
sfc.Rad_anomaly_planck_surf(experiment, kernel, cart_out)
sfc.Rad_anomaly_cloud(experiment, cart_out)

In [ ]:
# Compute anomalies all together and save them in the specified folder
sfc.calc_anoms(experiment, control, kernel, cart_out)

## Compute feedbacks

In [ ]:
# Compute single feedbacks and save them in the specified folder
name='albedo'
alb= sfc.single_feedback(name, experiment, kernel, cart_out)

In [ ]:
# Compute all feedbacks and save them in the specified folder
fb=sfc.calc_fb(experiment, control, kernel, cart_out)

In [ ]:
# Compute interannual feedbacks and save them in the specified folder
fb_interannual = sfc.calc_fb_interannual(experiment, control, kernel, cart_out)

## Save output and plot example

In [ ]:
out_path_txt='path/to/save/file_fb.txt'
out.save_feedback_output(fb, out_path_txt)

In [ ]:
feedback_file=out_path_txt
out.plot_single_feedback_file(feedback_file)